## Collobert & Weston embeddings implementations



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class NLPEmbeddingModel(nn.Module):
    """
    Deep Neural Network for NLP word embeddings based on the paper:
    "A Unified Architecture for Natural Language Processing: Deep Neural Networks with Multitask Learning"
    
    This implementation follows the architecture described in the paper, featuring:
    - Lookup table layers for word embeddings
    - Time-Delay Neural Network (TDNN) for sequence processing
    - Support for additional features like capitalization
    - Configurable dimensions and parameters
    
    The architecture is designed to learn meaningful representations of words in context,
    capturing both local and global syntactic/semantic patterns.
    """
    
    def __init__(self, vocab_size, embedding_dim, hidden_dim=100, kernel_size=3, use_capitalization=False):
        """
        Initialize the NLP embedding model based on the paper's architecture.
        
        Args:
            vocab_size (int): Size of the vocabulary (number of unique words)
            embedding_dim (int): Dimension of word embeddings (wsz in paper) - typically 15, 50, or 100
            hidden_dim (int): Number of hidden units in TDNN layer - controls model capacity
            kernel_size (int): Size of convolution kernel in TDNN - controls context window
            use_capitalization (bool): Whether to include capitalization as additional feature
        """
        super(NLPEmbeddingModel, self).__init__()
        
        self.embedding_dim = embedding_dim
        self.use_capitalization = use_capitalization
        
        # Lookup Table Layer (Section 3.1 in paper)
        # Maps discrete word indices to continuous vector representations
        # This is the fundamental building block that learns word embeddings
        self.word_embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # Capitalization embedding (Section 3.1 - Variations on Word Representations)
        # Adds information about word casing as an additional feature
        # This helps with tasks like Named Entity Recognition where capitalization matters
        if use_capitalization:
            self.cap_embedding = nn.Embedding(2, 2)  # 2 classes: capitalized or not
        
        # Time-Delay Neural Network Layer (Section 3.2 - Variable Sentence Length)
        # This is a 1D convolutional layer that processes sequences of word embeddings
        # It captures local context and dependencies between nearby words
        # The kernel_size determines how many adjacent words are considered together
        self.tdnn = nn.Conv1d(
            in_channels=embedding_dim + (2 if use_capitalization else 0),  # Input channels
            out_channels=hidden_dim,  # Number of feature maps
            kernel_size=kernel_size,  # Context window size
            padding=(kernel_size - 1) // 2  # Maintain sequence length
        )
        
        # Additional Hidden Layer (Section 3.3 - Deep Architecture)
        # Adds non-linearity and enables learning of more complex features
        # This is crucial for complex tasks like Semantic Role Labeling
        self.hidden = nn.Linear(hidden_dim, hidden_dim)
        
        # Initialize weights using Xavier uniform initialization
        # This helps with training stability and convergence
        self._init_weights()
    
    def _init_weights(self):
        """
        Initialize model weights using Xavier uniform initialization.
        
        This initialization method is particularly effective for layers with
        tanh or sigmoid activations, helping to maintain stable gradients
        during backpropagation through the deep network.
        """
        for module in self.modules():
            if isinstance(module, (nn.Linear, nn.Conv1d, nn.Embedding)):
                nn.init.xavier_uniform_(module.weight)
                if hasattr(module, 'bias') and module.bias is not None:
                    nn.init.constant_(module.bias, 0)
    
    def forward(self, word_ids, cap_features=None):
        """
        Forward pass of the neural network.
        
        Args:
            word_ids (torch.Tensor): Tensor of word indices [batch_size, seq_len]
            cap_features (torch.Tensor, optional): Tensor of capitalization features [batch_size, seq_len]
        
        Returns:
            torch.Tensor: Contextual word embeddings [batch_size, seq_len, hidden_dim]
            
        The forward pass follows this sequence:
        1. Word embedding lookup
        2. Optional capitalization feature concatenation
        3. TDNN convolution over sequence
        4. Non-linear hidden layer transformation
        """
        batch_size, seq_len = word_ids.shape
        
        # Step 1: Word Embedding Lookup (Section 3.1)
        # Convert discrete word indices to continuous vector representations
        # Each word becomes a dense vector in embedding_dim-dimensional space
        word_embeds = self.word_embedding(word_ids)  # [batch_size, seq_len, embedding_dim]
        
        # Step 2: Feature Concatenation (Section 3.1 - Variations on Word Representations)
        # Combine word embeddings with additional features like capitalization
        # This provides the model with more linguistic information
        if self.use_capitalization and cap_features is not None:
            cap_embeds = self.cap_embedding(cap_features)  # [batch_size, seq_len, 2]
            combined_embeds = torch.cat([word_embeds, cap_embeds], dim=-1)
        else:
            combined_embeds = word_embeds
        
        # Step 3: TDNN Processing (Section 3.2 - Variable Sentence Length)
        # Apply 1D convolution to capture sequential patterns and dependencies
        # The TDNN processes the entire sequence, considering local contexts
        # Transpose for Conv1d: [batch_size, channels, seq_len]
        tdnn_input = combined_embeds.transpose(1, 2)
        tdnn_output = self.tdnn(tdnn_input)  # [batch_size, hidden_dim, seq_len]
        tdnn_output = F.relu(tdnn_output)  # Apply non-linearity
        
        # Transpose back to original format: [batch_size, seq_len, hidden_dim]
        tdnn_output = tdnn_output.transpose(1, 2)
        
        # Step 4: Hidden Layer Transformation (Section 3.3 - Deep Architecture)
        # Additional non-linear transformation to learn complex features
        # This enables the model to capture higher-level abstractions
        final_embeddings = self.hidden(tdnn_output)  # [batch_size, seq_len, hidden_dim]
        final_embeddings = F.relu(final_embeddings)  # Apply non-linearity
        
        return final_embeddings


class Vocabulary:
    """
    Vocabulary class for mapping between words and integer indices.
    
    This class handles the conversion of raw text words to numerical indices
    that can be processed by the neural network. It follows the paper's approach
    of using a finite dictionary with an unknown token for out-of-vocabulary words.
    """
    
    def __init__(self, words_list):
        """
        Initialize vocabulary from a list of words.
        
        Args:
            words_list (list): List of words to include in vocabulary
        """
        self.word2idx = {}
        self.idx2word = {}
        self.unknown_token = "<UNK>"  # Special token for unknown words
        
        # Build vocabulary mappings
        # Include unknown token at index 0, then all vocabulary words
        words = [self.unknown_token] + words_list
        for idx, word in enumerate(words):
            self.word2idx[word] = idx
            self.idx2word[idx] = word
        
        self.vocab_size = len(words)
    
    def encode(self, words):
        """
        Convert a list of words to their corresponding indices.
        
        Args:
            words (list): List of words to encode
            
        Returns:
            list: List of integer indices
        """
        return [self.word2idx.get(word, self.word2idx[self.unknown_token]) for word in words]
    
    def decode(self, indices):
        """
        Convert a list of indices back to words.
        
        Args:
            indices (list): List of integer indices
            
        Returns:
            list: List of words
        """
        return [self.idx2word.get(idx, self.unknown_token) for idx in indices]


def preprocess_sentence(sentence, vocab, lowercase=True):
    """
    Preprocess a raw sentence and convert it to tensor format.
    
    This function implements the preprocessing steps described in the paper:
    - Tokenization (splitting into words)
    - Lowercasing (as mentioned in Section 3.1)
    - Capitalization feature extraction
    - Conversion to numerical indices
    
    Args:
        sentence (str): Raw input sentence
        vocab (Vocabulary): Vocabulary object for encoding
        lowercase (bool): Whether to convert to lowercase (paper uses True)
    
    Returns:
        dict: Dictionary containing word_ids and capitalization features as tensors
    """
    # Tokenize the sentence into individual words
    words = sentence.strip().split()
    
    # Preprocess each word in the sentence
    processed_words = []
    cap_features = []
    
    for word in words:
        # Extract capitalization information (Section 3.1)
        # Capitalization can be important for tasks like Named Entity Recognition
        is_capitalized = word[0].isupper() if word else 0
        
        # Apply lowercasing if specified (paper converts all words to lower case)
        if lowercase:
            processed_word = word.lower()
        else:
            processed_word = word
        
        processed_words.append(processed_word)
        cap_features.append(1 if is_capitalized else 0)
    
    # Convert words to numerical indices using vocabulary
    word_ids = vocab.encode(processed_words)
    
    # Return as tensors suitable for neural network input
    return {
        'word_ids': torch.tensor([word_ids], dtype=torch.long),
        'cap_features': torch.tensor([cap_features], dtype=torch.long) if cap_features else None
    }


def create_simple_vocabulary(sentences, vocab_size=30000):
    """
    Create a vocabulary from a collection of sentences.
    
    This follows the paper's approach of using the most frequent words
    from a large corpus (they used Wikipedia with 30,000 most common words).
    
    Args:
        sentences (list): List of sentences to build vocabulary from
        vocab_size (int): Maximum size of vocabulary (paper uses 30,000)
    
    Returns:
        Vocabulary: Initialized vocabulary object
    """
    from collections import Counter
    
    # Count word frequencies across all sentences
    word_counts = Counter()
    for sentence in sentences:
        words = sentence.lower().split()
        word_counts.update(words)
    
    # Select the most frequent words up to vocab_size
    # This prioritizes common words and discards rare ones
    most_common_words = [word for word, count in word_counts.most_common(vocab_size)]
    
    return Vocabulary(most_common_words)


def transform_sentence_to_vectors(sentence, embedding_dim=50, vocab=None, model_path=None):
    """
    Transform a raw sentence into contextual word vectors using the paper's architecture.
    
    This is the main function that implements the complete pipeline:
    1. Preprocessing and tokenization
    2. Vocabulary mapping
    3. Neural network forward pass
    4. Return contextual embeddings
    
    Args:
        sentence (str): Input sentence to transform
        embedding_dim (int): Desired embedding dimension (paper uses 15, 50, 100)
        vocab (Vocabulary): Vocabulary object (created automatically if None)
        model_path (str): Path to pre-trained model weights (uses random if None)
    
    Returns:
        torch.Tensor: Contextual word embeddings for the sentence [seq_len, hidden_dim]
    """
    # Create vocabulary if not provided
    # In practice, this would be created from a large corpus like Wikipedia
    if vocab is None:
        # For demonstration, create a small vocabulary
        sample_words = ["the", "cat", "sat", "on", "mat", "dog", "ran", "fast"]
        vocab = Vocabulary(sample_words)
    
    # Preprocess the sentence: tokenize, lowercase, extract features
    processed = preprocess_sentence(sentence, vocab, lowercase=True)
    
    # Initialize the neural network model
    model = NLPEmbeddingModel(
        vocab_size=vocab.vocab_size,
        embedding_dim=embedding_dim,
        hidden_dim=100,  # Paper uses various sizes depending on task
        kernel_size=3,   # Typical context window size
        use_capitalization=True  # Include capitalization features
    )
    
    # Load pre-trained weights if available
    # The paper shows that training on large unlabeled data (Wikipedia) gives better embeddings
    if model_path and os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path))
        print("Loaded pre-trained model weights")
    else:
        print("Using randomly initialized model (for demonstration)")
    
    # Set model to evaluation mode (disables dropout, batch norm, etc.)
    model.eval()
    
    # Generate embeddings through forward pass
    # torch.no_grad() disables gradient computation for efficiency
    with torch.no_grad():
        embeddings = model(
            processed['word_ids'], 
            processed['cap_features']
        )
    
    # Return embeddings for the first (and only) sentence in batch
    # Shape: [sequence_length, hidden_dimension]
    return embeddings[0]


# Example usage and demonstration
if __name__ == "__main__":
    """
    Demonstration of the complete word embedding pipeline.
    
    This example shows how to:
    1. Create a vocabulary from sample sentences
    2. Transform a raw sentence into contextual word embeddings
    3. Examine the resulting embeddings
    """
    
    # Sample sentences for building vocabulary
    sentences = [
        "The cat sat on the mat",
        "A dog ran very fast",
        "Natural language processing is amazing",
        "Deep neural networks learn representations"
    ]
    
    # Create vocabulary from sample sentences
    # In real applications, this would use millions of sentences
    vocab = create_simple_vocabulary(sentences, vocab_size=1000)
    
    # Transform a sentence into word vectors
    sentence = "The quick brown fox jumps over the lazy dog"
    embeddings = transform_sentence_to_vectors(
        sentence=sentence,
        embedding_dim=50,  # Same as paper's middle setting
        vocab=vocab
    )
    
    # Display results
    print("=" * 60)
    print("DEEP NEURAL NETWORK WORD EMBEDDING DEMONSTRATION")
    print("Based on: 'A Unified Architecture for Natural Language Processing'")
    print("=" * 60)
    print(f"Input sentence: {sentence}")
    print(f"Number of words: {len(sentence.split())}")
    print(f"Embedding shape: {embeddings.shape}")
    print(f"Embedding dimension: {embeddings.shape[1]}")
    print("\nFirst few dimensions of first word embedding:")
    print(f"'{sentence.split()[0]}': {embeddings[0][:10].numpy()}...")
    print("\nThese embeddings capture syntactic and semantic information")
    print("that can be used for various NLP tasks as described in the paper.")